# Phase 4.1 — Model training and cross-validation

This notebook loads the labeled Kaggle training CSV only. The reusable training workflow creates a stratified holdout before fitting, evaluates Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting on the same stratified cross-validation folds from the training partition, then refits all candidates on that training partition. The holdout is not scored, used for model selection, or used to fit transformations. No training accuracy is used. Each estimator includes the Phase 3 feature engineering and preprocessing pipeline.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "config.yaml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing config.yaml and data/")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from config.paths import get_project_paths
from config.settings import load_config
from data.loader import load_csv
from models.train import train_models

CONFIG = load_config()
PROJECT_PATHS = get_project_paths(CONFIG)
LABELED_PATH = PROJECT_PATHS.labeled_dataset
if not LABELED_PATH.is_file():
    raise FileNotFoundError(f"Labeled training CSV not found: {LABELED_PATH}")

labeled_df = load_csv(LABELED_PATH)
print(f"Labeled source: {LABELED_PATH}")
print(f"Rows: {len(labeled_df)}; target: {CONFIG['dataset']['target_column']}")
print("The separate unlabeled Kaggle test CSV is not loaded.")

Labeled source: C:\Projects\loan_predection_system\data\raw\train_u6lujuX_CVtuZ9i.csv
Rows: 614; target: Loan_Status
The separate unlabeled Kaggle test CSV is not loaded.


In [2]:
training_result = train_models(labeled_df, config=CONFIG)
display(training_result.cv_results)
print(f"Training report: {training_result.report_path}")
print(f"CV result JSON: {training_result.results_path}")
print("Fitted candidate pipelines:")
for model_name, model_path in training_result.model_paths.items():
    print(f"- {model_name}: {model_path}")
print("Holdout rows reserved and not evaluated:", len(training_result.split.X_test))

,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,specificity_mean,specificity_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,logistic_regression,0.722944,0.068146,0.678501,0.073726,0.742666,0.092224,0.558495,0.117233,0.798876,0.044162,0.798507,0.079124,0.797273,0.053575
1,decision_tree,0.655679,0.059531,0.631222,0.064065,0.647779,0.075640,0.564946,0.146946,0.780898,0.047226,0.697498,0.099360,0.733135,0.058334
2,random_forest,0.745351,0.061091,0.684495,0.082418,0.778701,0.072447,0.520000,0.168151,0.797509,0.051591,0.848990,0.068069,0.820500,0.041639
3,gradient_boosting,0.767718,0.068179,0.681068,0.088274,0.734186,0.110192,0.447957,0.153232,0.785177,0.048649,0.914179,0.054203,0.844044,0.044213


Training report: C:\Projects\loan_predection_system\reports\model_training_report.md
CV result JSON: C:\Projects\loan_predection_system\artifacts\reports\model_cv_results.json
Fitted candidate pipelines:
- logistic_regression: C:\Projects\loan_predection_system\artifacts\models\logistic_regression-1.0.0.joblib
- decision_tree: C:\Projects\loan_predection_system\artifacts\models\decision_tree-1.0.0.joblib
- random_forest: C:\Projects\loan_predection_system\artifacts\models\random_forest-1.0.0.joblib
- gradient_boosting: C:\Projects\loan_predection_system\artifacts\models\gradient_boosting-1.0.0.joblib
Holdout rows reserved and not evaluated: 123


## Interpretation boundary

The cross-validation values are model-development estimates, not final holdout performance. All four candidates are retained; this phase does not declare a winner or tune a decision threshold. A later evaluation phase must define the selection protocol and use the reserved holdout only for final evaluation.